# 3.8 — Human-in-the-Loop

Autonomous agents are powerful — but some actions are risky:
- Deleting a file
- Sending an email
- Placing an order
- Modifying a database

**Human-in-the-loop (HITL)** pauses the agent and asks for human approval before continuing.

```
Agent plans action
       ↓
  [PAUSE — show human]
       ↓
  Human: Approve / Reject / Edit
       ↓
  Agent continues (or stops)
```

In [ ]:
!pip install langchain langchain-ollama langgraph --quiet

## Step 1 — Define Risky Tools

In [ ]:
from langchain_core.tools import tool

# Safe tools — no approval needed
@tool
def get_account_balance(account_id: str) -> str:
    """Returns the current balance of a bank account."""
    balances = {'ACC001': '$5,420.00', 'ACC002': '$12,800.50', 'ACC003': '$890.25'}
    return balances.get(account_id, 'Account not found.')

@tool
def get_transaction_history(account_id: str) -> str:
    """Returns recent transactions for an account."""
    return f'Recent transactions for {account_id}: Coffee $4.50, Groceries $67.20, Netflix $15.99'

# Risky tools — require human approval
@tool
def transfer_money(from_account: str, to_account: str, amount: float) -> str:
    """Transfers money between two accounts. REQUIRES human approval."""
    return f'SUCCESS: Transferred ${amount:.2f} from {from_account} to {to_account}.'

@tool
def delete_account(account_id: str) -> str:
    """Permanently deletes a bank account. REQUIRES human approval."""
    return f'SUCCESS: Account {account_id} has been permanently deleted.'

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Sends an email to a recipient. REQUIRES human approval."""
    return f'SUCCESS: Email sent to {to} with subject "{subject}".'

# Mark which tools need approval
RISKY_TOOLS = {'transfer_money', 'delete_account', 'send_email'}

all_tools = [get_account_balance, get_transaction_history, transfer_money, delete_account, send_email]
tool_map  = {t.name: t for t in all_tools}

print('Tools defined.')
print(f'Safe tools  : {[t.name for t in all_tools if t.name not in RISKY_TOOLS]}')
print(f'Risky tools : {list(RISKY_TOOLS)}')

## Approach 1 — Manual Approval Check

The simplest approach: intercept tool calls and ask for confirmation before running risky ones.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

llm = ChatOllama(model='llama3.1', temperature=0)
llm_with_tools = llm.bind_tools(all_tools)

SYSTEM = """You are a helpful banking assistant.
You can check balances, view transaction history, transfer money, or send emails.
Always confirm the details before performing any financial action."""

def run_agent_with_approval(question: str) -> str:
    print(f'User: {question}')
    messages = [SystemMessage(content=SYSTEM), HumanMessage(content=question)]

    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            print(f'Agent: {response.content}\n')
            return response.content

        for tc in response.tool_calls:
            tool_name = tc['name']
            tool_args = tc['args']

            if tool_name in RISKY_TOOLS:
                # Show the proposed action to the human
                print(f'\n⚠️  Agent wants to call: {tool_name}')
                print(f'   Arguments: {tool_args}')
                approval = input('   Approve? (yes/no): ').strip().lower()

                if approval != 'yes':
                    result = f'Action {tool_name} was REJECTED by the user.'
                    print(f'   → Rejected.\n')
                else:
                    result = tool_map[tool_name].invoke(tool_args)
                    print(f'   → Approved. Result: {result}\n')
            else:
                # Safe tool — run without asking
                result = tool_map[tool_name].invoke(tool_args)
                print(f'  [auto] {tool_name}({tool_args}) → {result}')

            messages.append(ToolMessage(content=str(result), tool_call_id=tc['id']))

print('Agent with approval ready.')

In [ ]:
# Safe action — no approval prompt
run_agent_with_approval('What is the balance of account ACC001?')

In [ ]:
# Risky action — will pause and ask for approval
# Type 'yes' to approve or 'no' to reject
run_agent_with_approval('Transfer $500 from ACC001 to ACC002')

## Approach 2 — LangGraph with Interrupt

LangGraph has a built-in `interrupt` mechanism — the graph pauses at a node and waits for human input to resume.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langgraph.types import interrupt, Command
from langchain_core.messages import BaseMessage, HumanMessage
from typing import TypedDict, Annotated, Literal
import operator

class BankState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

# Separate safe and risky tools into separate ToolNodes
safe_tools  = [get_account_balance, get_transaction_history]
risky_tools = [transfer_money, delete_account, send_email]

safe_tool_node  = ToolNode(safe_tools)
risky_tool_node = ToolNode(risky_tools)

llm_all = llm.bind_tools(safe_tools + risky_tools)

def agent_node(state: BankState) -> BankState:
    response = llm_all.invoke([SystemMessage(content=SYSTEM)] + state['messages'])
    return {'messages': [response]}

def route_tools(state: BankState) -> Literal['safe_tools', 'human_review', '__end__']:
    last = state['messages'][-1]
    if not hasattr(last, 'tool_calls') or not last.tool_calls:
        return '__end__'
    # Route risky tools to human_review, safe tools directly
    for tc in last.tool_calls:
        if tc['name'] in {t.name for t in risky_tools}:
            return 'human_review'
    return 'safe_tools'

def human_review_node(state: BankState) -> Command:
    """Interrupt graph and show proposed action to human."""
    last = state['messages'][-1]
    tool_calls = last.tool_calls

    print('\n⚠️  HUMAN REVIEW REQUIRED')
    for tc in tool_calls:
        print(f'  Tool     : {tc["name"]}')
        print(f'  Arguments: {tc["args"]}')

    decision = interrupt('Approve this action? (yes/no): ')

    if decision.lower() == 'yes':
        return Command(goto='risky_tools')
    else:
        # Inject rejection message
        rejection = [ToolMessage(
            content='Action rejected by user.',
            tool_call_id=tool_calls[0]['id']
        )]
        return Command(goto='agent', update={'messages': rejection})

# Build graph
checkpointer = MemorySaver()

bank_graph = StateGraph(BankState)
bank_graph.add_node('agent',        agent_node)
bank_graph.add_node('safe_tools',   safe_tool_node)
bank_graph.add_node('risky_tools',  risky_tool_node)
bank_graph.add_node('human_review', human_review_node)

bank_graph.add_edge(START,         'agent')
bank_graph.add_conditional_edges('agent', route_tools)
bank_graph.add_edge('safe_tools',  'agent')
bank_graph.add_edge('risky_tools', 'agent')

bank_app = bank_graph.compile(checkpointer=checkpointer)

print('LangGraph HITL agent compiled.')

In [ ]:
import uuid

def run_with_hitl(question: str):
    thread = {'configurable': {'thread_id': str(uuid.uuid4())}}
    print(f'User: {question}\n')

    result = bank_app.invoke(
        {'messages': [HumanMessage(content=question)]},
        config=thread
    )

    # Check if interrupted
    state = bank_app.get_state(thread)
    if state.next:
        print(f'\nGraph paused. Waiting for human input at: {state.next}')
        approval = input('Your decision (yes/no): ').strip()
        # Resume with human's decision
        final = bank_app.invoke(Command(resume=approval), config=thread)
        print(f"\nAgent: {final['messages'][-1].content}")
    else:
        print(f"Agent: {result['messages'][-1].content}")

# Safe: no interruption
run_with_hitl('Show me the balance of ACC001')

In [ ]:
# Risky: will interrupt and ask for approval
run_with_hitl('Please transfer $200 from ACC001 to ACC002')

## Summary

| Approach | How it works | Best for |
|----------|-------------|----------|
| **Manual check** | Python `if` before calling risky tool | Simple scripts |
| **LangGraph interrupt** | Graph pauses at node, resumes after input | Production apps with persistence |

**Always add HITL for:**
- Financial transactions
- Sending messages to external systems
- Deleting or modifying persistent data
- Any irreversible action